# search4.0 含自回归特征 σ_t 的全自动化随机探索

> **说明**：本 Notebook 产出 `search4.0_global_summary_metrics.csv` 与 `tune_v2_*.csv`，供超参搜索与敏感性分析归档。**论文正文 RMSE/MAE/R² 以 `results/final_model_summary_metrics_v2.csv` 为唯一口径**；出图勿用 search4.0 的 `Test_RMSE` 替代定型表。代码保留，定型指标表由历史定型训练固化。

与 search3.0 完全一致的方法论（统一超参数空间、n_iter=30、seed=42），
唯一区别：特征矩阵使用 `04_final_feature_matrix_v2.csv`，G1 基础组新增 σ_t（当前日波动率）作为自回归特征。

In [1]:
import os, copy, time, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterSampler
from scipy.stats import loguniform, uniform
from IPython.display import display

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'>>> 当前计算设备: {device}')

>>> 当前计算设备: cuda


## 1. 全局配置与 Dataset 定义

In [2]:
DATA_PATH = '../data/04_final_feature_matrix_v2.csv'
RESULT_DIR = '../results/'
os.makedirs(RESULT_DIR, exist_ok=True)

class CryptoVolatilityDataset(Dataset):
    def __init__(self, feature_matrix, target_array, lookback_window=7):
        self.features = feature_matrix
        self.targets = target_array
        self.window = lookback_window
    def __len__(self):
        return len(self.features) - self.window + 1
    def __getitem__(self, idx):
        x = self.features[idx : idx + self.window]
        y = self.targets[idx + self.window - 1]
        return torch.FloatTensor(x), torch.FloatTensor([y])

## 2. Baseline(LSTM) 与 Main(BiLSTM+Attention) 网络架构

In [3]:
class BaselineLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p):
        super(BaselineLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)
        self.fc = nn.Sequential(nn.Linear(hidden_size, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

class AttentionLayer(nn.Module):
    def __init__(self, hidden_dim):
        super(AttentionLayer, self).__init__()
        self.attention_weights = nn.Linear(hidden_dim, 1, bias=False)
    def forward(self, lstm_output):
        scores = self.attention_weights(lstm_output)
        alpha = F.softmax(scores, dim=1)
        context_vector = torch.sum(alpha * lstm_output, dim=1)
        return context_vector, alpha.squeeze(-1)

class BiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p):
        super(BiLSTMAttention, self).__init__()
        self.bilstm = nn.LSTM(input_size, hidden_size, num_layers=1, batch_first=True, bidirectional=True)
        lstm_out_dim = hidden_size * 2
        self.attention = AttentionLayer(hidden_dim=lstm_out_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc_layers = nn.Sequential(nn.Linear(lstm_out_dim, 32), nn.ReLU(), nn.Linear(32, 1))
    def forward(self, x):
        lstm_out, _ = self.bilstm(x)
        context, attn_weights = self.attention(lstm_out)
        dropped_out = self.dropout(context)
        out = self.fc_layers(dropped_out)
        return out, attn_weights

## 3. 核心封装：`tune_model`

In [4]:
def _train_and_validate(model, train_loader, val_loader, params, is_attention=False):
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=7, min_lr=1e-6)
    best_val_loss, best_model_wts = float('inf'), None
    patience, epochs_no_improve, max_epochs = 20, 0, 300
    for epoch in range(max_epochs):
        model.train()
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(x_batch)[0] if is_attention else model(x_batch)
            loss = criterion(preds, y_batch)
            loss.backward(); optimizer.step()
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_v, y_v in val_loader:
                x_v, y_v = x_v.to(device), y_v.to(device)
                preds = model(x_v)[0] if is_attention else model(x_v)
                val_losses.append(criterion(preds, y_v).item())
        avg_val_loss = np.mean(val_losses)
        scheduler.step(avg_val_loss)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience: break
    if best_model_wts is not None: model.load_state_dict(best_model_wts)
    return model, best_val_loss


def tune_model(model_type, feature_group):
    assert model_type in ['Baseline', 'Main']
    is_attention = (model_type == 'Main')
    model_class = BiLSTMAttention if is_attention else BaselineLSTM
    
    df = pd.read_csv(DATA_PATH)
    all_cols = list(df.columns)
    base_features = [c for c in all_cols if c not in [
        'timestamp', 'target_sigma_t_plus_1', 'finbert_sentiment_score',
        'llm_composite_score', 'sentiment_class', 'action_class', 'action_score']]
    
    fg_map = {
        'G1': base_features,
        'G2': base_features + ['finbert_sentiment_score'],
        'G3': base_features + ['llm_composite_score'],
        'G3-M': base_features + ['sentiment_class', 'action_class', 'action_score']
    }
    f_cols = fg_map[feature_group]
    train_end, val_end = int(len(df)*0.7), int(len(df)*0.8)
    
    train_ds = CryptoVolatilityDataset(df.iloc[:train_end][f_cols].values, df.iloc[:train_end]['target_sigma_t_plus_1'].values)
    val_ds = CryptoVolatilityDataset(df.iloc[train_end:val_end][f_cols].values, df.iloc[train_end:val_end]['target_sigma_t_plus_1'].values)
    test_ds = CryptoVolatilityDataset(df.iloc[val_end:][f_cols].values, df.iloc[val_end:]['target_sigma_t_plus_1'].values)
    
    param_dist = {
        'hidden_size': [64, 128, 256],
        'dropout': uniform(0.2, 0.2),
        'batch_size': [32, 64],
        'lr': loguniform(5e-4, 5e-3),
        'weight_decay': [1e-5, 5e-5, 1e-4]
    }
    n_iter = 30
    sampler = ParameterSampler(param_dist, n_iter=n_iter, random_state=SEED)
    combinations = list(sampler)
    
    model_save_dir = f'../models/tune_v2_{model_type}_{feature_group}/'
    os.makedirs(model_save_dir, exist_ok=True)
    log_file = os.path.join(RESULT_DIR, f'tune_v2_{model_type}_{feature_group}.csv')
    
    print(f'\n{"="*50}')
    print(f' 🚀 开始 {model_type} 模型的 Randomized Search')
    print(f' 🎯 特征组: {feature_group} | 特征维度: {len(f_cols)} 维 | 探索空间: {n_iter} 轮')
    print('='*50)
    
    best_overall_val_loss, best_overall_model_state, best_overall_params = float('inf'), None, None
    
    for i, params in enumerate(combinations):
        h, dp, bs = params['hidden_size'], round(params['dropout'], 4), params['batch_size']
        lr, wd = round(params['lr'], 5), params['weight_decay']
        print(f'[{i+1}/{n_iter}] H:{h:3d} | DP:{dp:.4f} | BS:{bs:2d} | LR:{lr:.5f} | WD:{wd:.5f}', end='')
        
        train_loader = DataLoader(train_ds, batch_size=bs, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=bs, shuffle=False)
        torch.cuda.empty_cache()
        model = model_class(len(f_cols), h, params['dropout']).to(device)
        model, best_val_loss = _train_and_validate(model, train_loader, val_loader, params, is_attention=is_attention)
        print(f'  => Val Loss: {best_val_loss:.6f}')
        
        if best_val_loss < best_overall_val_loss:
            best_overall_val_loss = best_val_loss
            best_overall_model_state = copy.deepcopy(model.state_dict())
            best_overall_params = params
        
        result_entry = {'Model': model_type, 'FeatureGroup': feature_group,
            'hidden_size': h, 'dropout': dp, 'batch_size': bs, 'lr': lr, 'weight_decay': wd,
            'Val_Loss': best_val_loss, 'Timestamp': time.strftime('%Y-%m-%d %H:%M:%S')}
        pd.DataFrame([result_entry]).to_csv(log_file, mode='a', header=not os.path.exists(log_file), index=False)
    
    print(f'\n🏆 [{model_type} - {feature_group}] 搜索完毕！')
    print(f'🔥 最佳验证集 Loss: {best_overall_val_loss:.6f}')
    
    best_model = model_class(len(f_cols), best_overall_params['hidden_size'], best_overall_params['dropout']).to(device)
    best_model.load_state_dict(best_overall_model_state)
    best_model.eval()
    
    test_loader = DataLoader(test_ds, batch_size=best_overall_params['batch_size'], shuffle=False)
    preds, targets = [], []
    with torch.no_grad():
        for x_t, y_t in test_loader:
            batch_preds = best_model(x_t.to(device))[0] if is_attention else best_model(x_t.to(device))
            preds.extend(batch_preds.cpu().numpy()); targets.extend(y_t.numpy())
    preds, targets = np.array(preds).flatten(), np.array(targets).flatten()
    rmse = np.sqrt(mean_squared_error(targets, preds))
    mae = mean_absolute_error(targets, preds)
    r2 = r2_score(targets, preds)
    
    print(f'====================================================')
    print(f'📊 测试集 (N={len(targets)}) 最终指标: RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}')
    print(f'====================================================\n')
    
    torch.save(best_overall_model_state, os.path.join(model_save_dir, 'best_overall_model.pth'))
    return best_overall_params, rmse, mae, r2

## 4. 全量自动化遍历执行

In [5]:
all_results = []
for model_type in ['Baseline', 'Main']:
    for fg in ['G1', 'G2', 'G3', 'G3-M']:
        best_params, rmse, mae, r2 = tune_model(model_type, fg)
        all_results.append({
            'Model': model_type, 'Feature_Group': fg,
            'Test_RMSE': rmse, 'Test_MAE': mae, 'Test_R2': r2,
            'Best_Hidden': best_params['hidden_size'],
            'Best_Dropout': round(best_params['dropout'], 4),
            'Best_BatchSize': best_params['batch_size'],
            'Best_LR': round(best_params['lr'], 5),
            'Best_WD': best_params['weight_decay']
        })

summary_df = pd.DataFrame(all_results)
summary_path = os.path.join(RESULT_DIR, 'search4.0_global_summary_metrics.csv')
summary_df.to_csv(summary_path, index=False)

print('\n' + '='*60)
print('🎉 所有模型与特征组的 Randomized Search (v2 含 σ_t) 全部执行完毕！')
print('='*60)
display(summary_df)


 🚀 开始 Baseline 模型的 Randomized Search
 🎯 特征组: G1 | 特征维度: 21 维 | 探索空间: 30 轮
[1/30] H:256 | DP:0.3593 | BS:32 | LR:0.00270 | WD:0.00001  => Val Loss: 1.697366
[2/30] H:256 | DP:0.2312 | BS:32 | LR:0.00063 | WD:0.00010  => Val Loss: 1.789736
[3/30] H:256 | DP:0.2667 | BS:64 | LR:0.00052 | WD:0.00005  => Val Loss: 1.709155
[4/30] H:128 | DP:0.3665 | BS:64 | LR:0.00050 | WD:0.00001  => Val Loss: 3.197252
[5/30] H:128 | DP:0.2608 | BS:32 | LR:0.00051 | WD:0.00001  => Val Loss: 2.195212
[6/30] H:256 | DP:0.3050 | BS:32 | LR:0.00069 | WD:0.00010  => Val Loss: 1.718225
[7/30] H:128 | DP:0.2181 | BS:64 | LR:0.00079 | WD:0.00010  => Val Loss: 2.092859
[8/30] H:256 | DP:0.3185 | BS:64 | LR:0.00362 | WD:0.00010  => Val Loss: 2.143156
[9/30] H:128 | DP:0.2901 | BS:32 | LR:0.00444 | WD:0.00005  => Val Loss: 1.859978
[10/30] H:128 | DP:0.2771 | BS:64 | LR:0.00063 | WD:0.00010  => Val Loss: 2.149363
[11/30] H: 64 | DP:0.2244 | BS:64 | LR:0.00341 | WD:0.00010  => Val Loss: 2.193190
[12/30] H:128 | DP:0.

,Model,Feature_Group,Test_RMSE,Test_MAE,Test_R2,Best_Hidden,Best_Dropout,Best_BatchSize,Best_LR,Best_WD
0,Baseline,G1,1.560034,1.269529,-0.300637,256,0.2127,32,0.00233,0.00001
1,Baseline,G2,1.475639,1.191773,-0.163720,128,0.2472,64,0.00156,0.00001
2,Baseline,G3,2.561637,2.059460,-2.506899,256,0.2587,64,0.00327,0.00001
3,Baseline,G3-M,1.414283,1.097434,-0.068958,256,0.3302,32,0.00354,0.00005
4,Main,G1,1.846167,1.530237,-0.821503,64,0.3859,64,0.00099,0.00001
5,Main,G2,1.486451,1.201963,-0.180835,256,0.3886,64,0.00097,0.00005
6,Main,G3,1.330176,1.028453,0.054402,256,0.2587,64,0.00327,0.00001
7,Main,G3-M,1.616835,1.318164,-0.397073,256,0.2667,64,0.00052,0.00005
